### Download MELD Dataset

First, clone the MELD GitHub repository which contains the download script and other resources.

In [ ]:
!git clone https://github.com/declare-lab/MELD.git

Cloning into 'MELD'...
remote: Enumerating objects: 493, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 493 (delta 10), reused 12 (delta 5), pack-reused 475 (from 1)
Receiving objects: 100% (493/493), 8.12 MiB | 14.29 MiB/s, done.
Resolving deltas: 100% (258/258), done.


In [ ]:
!pip install transformers

In [ ]:
%%bash
ls -R MELD

MELD:
baseline
data
images
LICENSE
README.md
utils

MELD/baseline:
baseline.py
data_helpers.py

MELD/data:
emorynlp
MELD
MELD_Dyadic
pickles

MELD/data/emorynlp:
emorynlp_dev_final.csv
emorynlp_test_final.csv
emorynlp_train_final.csv

MELD/data/MELD:
datasets.yaml
dev_sent_emo.csv
test_sent_emo.csv
train_sent_emo.csv

MELD/data/MELD_Dyadic:
dev_sent_emo_dya.csv
test_sent_emo_dya.csv
train_sent_emo_dya.csv

MELD/data/pickles:
download-features.txt

MELD/images:
emotion_shift.jpeg
meld.png
sc4.png

MELD/utils:
read_emorynlp.py
read_meld.py


## Load Data

In [ ]:
import pandas as pd

# Load dataset into datafrane
train_df = pd.read_csv('MELD/data/MELD/train_sent_emo.csv')
test_df = pd.read_csv('MELD/data/MELD/test_sent_emo.csv')
val_df = pd.read_csv('MELD/data/MELD/dev_sent_emo.csv')

In [ ]:
train_df.head()

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,also I was the point person on my company’s tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731"
1,2,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442"
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389"
3,4,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572"
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917"


In [ ]:
test_df.head()

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,Why do all you’re coffee mugs have numbers on ...,Mark,surprise,positive,0,0,3,19,"00:14:38,127","00:14:40,378"
1,2,Oh. That’s so Monica can keep track. That way ...,Rachel,anger,negative,0,1,3,19,"00:14:40,629","00:14:47,385"
2,3,Y'know what?,Rachel,neutral,neutral,0,2,3,19,"00:14:56,353","00:14:57,520"
3,19,"Come on, Lydia, you can do it.",Joey,neutral,neutral,1,0,1,23,"0:10:44,769","0:10:46,146"
4,20,Push!,Joey,joy,positive,1,1,1,23,"0:10:46,146","0:10:46,833"


In [ ]:
val_df.head()

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,"Oh my God, he’s lost it. He’s totally lost it.",Phoebe,sadness,negative,0,0,4,7,"00:20:57,256","00:21:00,049"
1,2,What?,Monica,surprise,negative,0,1,4,7,"00:21:01,927","00:21:03,261"
2,3,"Or! Or, we could go to the bank, close our acc...",Ross,neutral,neutral,1,0,4,4,"00:12:24,660","00:12:30,915"
3,4,You’re a genius!,Chandler,joy,positive,1,1,4,4,"00:12:32,334","00:12:33,960"
4,5,"Aww, man, now we won’t be bank buddies!",Joey,sadness,negative,1,2,4,4,"00:12:34,211","00:12:37,505"


## Preprocessing

In [ ]:
def build_dialogues_from_df(df, context_window=3, labels_to_idx=None):
  temp_df = df.copy()
  temp_df["speaker_utterance"] = temp_df["Speaker"] + ": " + temp_df["Utterance"]

  if labels_to_idx is None:
      labels_to_idx = {label: idx for idx, label in enumerate(temp_df["Emotion"].unique())}
  temp_df["Emotion"] = temp_df["Emotion"].map(labels_to_idx)

  def build_context(group):
      group=group.sort_values(by=["Utterance_ID"])
      utterances = group["speaker_utterance"].tolist()
      contexts = []

      for i in range(len(utterances)):
          start = max(0, i - context_window)
          context = "[CLS] " + " [SEP] ".join(utterances[start:i+1][::-1])
          contexts.append(context)

      group["dialogue"] = contexts
      return group

  temp_df = temp_df.groupby("Dialogue_ID", group_keys=False).apply(build_context, include_groups=False)
  return temp_df[["dialogue", "Emotion"]].rename(columns={"Emotion": "label"}), labels_to_idx

def preprocess_dialogue_data(train, test, val, context_window=3):
  train_dialogues, train_labels = build_dialogues_from_df(train, context_window=context_window)
  test_dialogues, _ = build_dialogues_from_df(test, labels_to_idx=train_labels, context_window=context_window)
  val_dialogues, _ = build_dialogues_from_df(val, labels_to_idx=train_labels, context_window=context_window)
  return train_dialogues, test_dialogues, val_dialogues, train_labels

In [ ]:
train_dialogues, test_dialogues, val_dialogues, labels_to_idx = preprocess_dialogue_data(train_df, test_df, val_df, context_window=3)

In [ ]:
train_dialogues.iloc[12:16]

,dialogue,label
12,[CLS] Chandler: Really?! [SEP] The Interviewer...,1
13,[CLS] The Interviewer: Absolutely. You can re...,0
14,[CLS] Joey: But then who? The waitress I went ...,1
15,[CLS] Rachel: You know? Forget it! [SEP] Joey:...,3


In [ ]:
labels_to_idx

{'neutral': 0,
 'surprise': 1,
 'fear': 2,
 'sadness': 3,
 'joy': 4,
 'disgust': 5,
 'anger': 6}

### Train model

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
from transformers import BertTokenizer, BertModel

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert = BertModel.from_pretrained(model_name)
def encode_texts(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
        # truncation_side="left"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch
import torch.nn as nn
from time import time

class BertClassifier(nn.Module):
    def __init__(self, num_classes, unfreezed_last=1):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)

        # Freeze most of BERT
        for name, param in self.bert.named_parameters():
            param.requires_grad = False

        # Unfreeze last n layers
        for name, param in self.bert.named_parameters():
            if "encoder.layer" in name:
                layer_num = int(name.split(".")[2])
                if layer_num >= 12 - unfreezed_last and "attention" in name:
                    param.requires_grad = True

        if "pooler" in name:
            param.requires_grad = True

        self.dropout = nn.Dropout(0.5)

        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, num_classes)
        )

        self.attention = nn.Linear(768, 1)

    # def forward(self, input_ids, attention_mask):
    #     outputs = self.bert(
    #         input_ids=input_ids,
    #         attention_mask=attention_mask
    #     )

    #     mask = attention_mask.unsqueeze(-1)
    #     masked_output = outputs.last_hidden_state * mask

    #     mean_output = masked_output.sum(dim=1) / mask.sum(dim=1)
    #     x = self.dropout(mean_output)

    #     # cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS]
    #     # x = self.dropout(cls_output)

    #     logits = self.classifier(x)

    #     return logits

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # (B, T, 768)
        hidden_states = outputs.last_hidden_state

        # Compute attention scores (B, T, 1)
        attn_scores = self.attention(hidden_states)

        # Mask padding tokens
        attn_scores = attn_scores.masked_fill(
            attention_mask.unsqueeze(-1) == 0, -1e9
        )

        # Normalize to get attention weights
        attn_weights = torch.softmax(attn_scores, dim=1)

        # Weighted sum of hidden states
        context_vector = (hidden_states * attn_weights).sum(dim=1)

        # Dropout + classifier
        x = self.dropout(context_vector)
        logits = self.classifier(x)

        return logits

In [ ]:
class DialogueDataset(torch.utils.data.Dataset):
    def __init__(self, dialogues):
        texts = dialogues["dialogue"].tolist()
        labels = dialogues["label"].tolist()
        encodings = encode_texts(texts)
        self.input_ids = [x for x in encodings["input_ids"]]
        self.attention_mask = [x for x in encodings["attention_mask"]]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        """
        alpha: Tensor of shape (num_classes,) or None
               → class weights (like CrossEntropyLoss weight)
        gamma: focusing parameter (default=2.0)
        reduction: 'mean', 'sum', or 'none'
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        """
        logits: (batch_size, num_classes)
        targets: (batch_size,)
        """

        targets = targets.to(device)
        logits = logits.to(device)
        # Standard cross entropy (no reduction)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')

        # pt = probability of correct class
        pt = torch.exp(-ce_loss)

        # Apply focal scaling
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        # Apply class weights if provided
        if self.alpha is not None:
            alpha = self.alpha.to(device)
            alpha_t = alpha[targets]  # pick weight per sample
            focal_loss = alpha_t * focal_loss

        # Reduction
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
from torch.utils.data import DataLoader

def get_loader(train, test, val, context_window=3, batch_size=32, weight_type: str="inverse"):
  train_dialogues, test_dialogues, val_dialogues, labels_to_idx = preprocess_dialogue_data(train, test, val, context_window=context_window)

  train_dataset = DialogueDataset(train_dialogues)
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

  class_counts = train_dialogues["label"].value_counts().sort_index()

  if weight_type == "inverse":
      weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float)
  elif weight_type == "inverse_sqrt":
      weights = 1.0 / torch.sqrt(torch.tensor(class_counts.values, dtype=torch.float))
  else:
      weights = torch.ones(len(labels_to_idx))

  weights = weights / weights.sum() * len(class_counts)

  test_dataset = DialogueDataset(test_dialogues)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

  val_dataset = DialogueDataset(val_dialogues)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
  return train_loader, test_loader, val_loader, weights

In [ ]:
train_loader, test_loader, val_loader, weights = get_loader(train_df, test_df, val_df, context_window=1, batch_size=32, weight_type="inverse_sqrt")

In [ ]:
print(weights)

tensor([0.3962, 0.7833, 1.6609, 1.0404, 0.6513, 1.6516, 0.8165])


In [ ]:
from sklearn.metrics import f1_score, classification_report
from torch.utils.tensorboard import SummaryWriter
import os

for context_window in [0, 1, 3, 5]:
    train_loader, test_loader, val_loader, weights = get_loader(train_df, test_df, val_df, context_window=context_window, batch_size=32, weight_type="normal")
    print(f"Context window: {context_window}")

    # Initialize TensorBoard writer
    log_dir = f"runs/meld_experiment_cw_{context_window}"
    writer = SummaryWriter(log_dir=log_dir)

    model = BertClassifier(num_classes=len(labels_to_idx), unfreezed_last=11)
    model.to(device)

    optimizer = torch.optim.AdamW([
        {
            "params": [p for n, p in model.bert.named_parameters() if p.requires_grad],
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-4
        },
        {
            "params": model.dropout.parameters(),
            "lr": 1e-4
        },
        {
            "params": model.attention.parameters(),
            "lr": 1e-4
        },

    ], weight_decay=0.01)

    criterion = FocalLoss(alpha=weights,gamma=1.5)

    best_val_f1 = -1
    best_model_state = None
    best_val_preds = []
    best_val_labels = []

    for epoch in range(7):
        # ===================== TRAIN =====================
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1
            )
            optimizer.step()

            train_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_acc = train_correct / train_total
        train_loss = train_loss / train_total

        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Accuracy/train", train_acc, epoch)

        # ===================== VALIDATION =====================
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        current_val_preds = []
        current_val_labels = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)

                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                current_val_preds.extend(preds.cpu().numpy())
                current_val_labels.extend(labels.cpu().numpy())

        val_acc = val_correct / val_total
        val_loss = val_loss / val_total
        val_f1 = f1_score(current_val_labels, current_val_preds, average='weighted')

        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)
        writer.add_scalar("F1_score/val", val_f1, epoch)

        # Save best model and its validation predictions/labels
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()
            best_val_preds = current_val_preds
            best_val_labels = current_val_labels

        # ===================== LOG =====================
        print(
            f"Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} "
            f"Val F1: {val_f1:.4f}"
        )

    # Generate classification report for the best model on the validation set
    if best_model_state:
        print(f"\nBest Validation F1 score: {best_val_f1:.4f}")
        # Create a reverse mapping from index to label name
        idx_to_label = {idx: label for label, idx in labels_to_idx.items()}
        target_names = [idx_to_label[i] for i in sorted(idx_to_label.keys())]
        print("\nClassification Report (Validation Set):\n")
        print(classification_report(best_val_labels, best_val_preds, target_names=target_names))

    writer.close()

Context window: 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Train Loss: 1.0082, Train Acc: 0.5426 | Val Loss: 0.8752, Val Acc: 0.5825 Val F1: 0.5203
Epoch 2 | Train Loss: 0.8053, Train Acc: 0.6259 | Val Loss: 0.8104, Val Acc: 0.6087 Val F1: 0.5712
Epoch 3 | Train Loss: 0.7548, Train Acc: 0.6478 | Val Loss: 0.7969, Val Acc: 0.6159 Val F1: 0.5766
Epoch 4 | Train Loss: 0.7193, Train Acc: 0.6636 | Val Loss: 0.7889, Val Acc: 0.6177 Val F1: 0.5797
Epoch 5 | Train Loss: 0.6871, Train Acc: 0.6707 | Val Loss: 0.7882, Val Acc: 0.6204 Val F1: 0.5852
Epoch 6 | Train Loss: 0.6514, Train Acc: 0.6830 | Val Loss: 0.7685, Val Acc: 0.6141 Val F1: 0.5920
Epoch 7 | Train Loss: 0.6187, Train Acc: 0.6936 | Val Loss: 0.7974, Val Acc: 0.6177 Val F1: 0.5886

Best Validation F1 score: 0.5920

Classification Report (Validation Set):

              precision    recall  f1-score   support

     neutral       0.71      0.84      0.77       470
    surprise       0.62      0.61      0.62       150
        fear       0.40      0.10      0.16        40
     sadness  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Train Loss: 1.0800, Train Acc: 0.4898 | Val Loss: 0.9658, Val Acc: 0.5113 Val F1: 0.4538
Epoch 2 | Train Loss: 0.8388, Train Acc: 0.5978 | Val Loss: 0.8181, Val Acc: 0.5906 Val F1: 0.5492
Epoch 3 | Train Loss: 0.7514, Train Acc: 0.6410 | Val Loss: 0.7902, Val Acc: 0.6087 Val F1: 0.5752
Epoch 4 | Train Loss: 0.7051, Train Acc: 0.6598 | Val Loss: 0.7707, Val Acc: 0.6078 Val F1: 0.5730
Epoch 5 | Train Loss: 0.6679, Train Acc: 0.6725 | Val Loss: 0.7616, Val Acc: 0.6087 Val F1: 0.5815
Epoch 6 | Train Loss: 0.6321, Train Acc: 0.6883 | Val Loss: 0.7593, Val Acc: 0.6123 Val F1: 0.5849
Epoch 7 | Train Loss: 0.5965, Train Acc: 0.7023 | Val Loss: 0.7798, Val Acc: 0.6186 Val F1: 0.5889

Best Validation F1 score: 0.5889

Classification Report (Validation Set):

              precision    recall  f1-score   support

     neutral       0.69      0.87      0.77       470
    surprise       0.58      0.68      0.63       150
        fear       0.43      0.07      0.13        40
     sadness  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Train Loss: 1.1048, Train Acc: 0.4848 | Val Loss: 0.9751, Val Acc: 0.5212 Val F1: 0.4691
Epoch 2 | Train Loss: 0.8576, Train Acc: 0.5934 | Val Loss: 0.8175, Val Acc: 0.5861 Val F1: 0.5509
Epoch 3 | Train Loss: 0.7600, Train Acc: 0.6400 | Val Loss: 0.8012, Val Acc: 0.5996 Val F1: 0.5618
Epoch 4 | Train Loss: 0.7143, Train Acc: 0.6598 | Val Loss: 0.7822, Val Acc: 0.6096 Val F1: 0.5790
Epoch 5 | Train Loss: 0.6688, Train Acc: 0.6757 | Val Loss: 0.7877, Val Acc: 0.6032 Val F1: 0.5703
Epoch 6 | Train Loss: 0.6331, Train Acc: 0.6868 | Val Loss: 0.7468, Val Acc: 0.6186 Val F1: 0.5908
Epoch 7 | Train Loss: 0.5953, Train Acc: 0.7013 | Val Loss: 0.7655, Val Acc: 0.6168 Val F1: 0.5932

Best Validation F1 score: 0.5932

Classification Report (Validation Set):

              precision    recall  f1-score   support

     neutral       0.71      0.83      0.77       470
    surprise       0.61      0.71      0.65       150
        fear       0.38      0.07      0.12        40
     sadness  

In [ ]:
import os
# Ensure the log directory exists
if not os.path.exists('runs'):
    print("TensorBoard log directory 'runs' not found. Please ensure the training cell has been executed.")
else:
    # Launch TensorBoard
    %load_ext tensorboard
    %tensorboard --logdir runs

In [ ]:
# 7 full
# 7 attention inverse_sqrt

In [ ]:
#  Context window: 0
# Loading weights: 100% 199/199 [00:00<00:00, 690.45it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.5015, Train Acc: 0.5819 | Val Loss: 1.3589, Val Acc: 0.6159 Val F1: 0.5934
# Epoch 2 | Train Loss: 1.2941, Train Acc: 0.6363 | Val Loss: 1.3224, Val Acc: 0.6123 Val F1: 0.5999
# Epoch 3 | Train Loss: 1.1943, Train Acc: 0.6532 | Val Loss: 1.2980, Val Acc: 0.6159 Val F1: 0.6073
# Epoch 4 | Train Loss: 1.0922, Train Acc: 0.6794 | Val Loss: 1.3262, Val Acc: 0.6060 Val F1: 0.6020
# Epoch 5 | Train Loss: 0.9853, Train Acc: 0.7035 | Val Loss: 1.3601, Val Acc: 0.5870 Val F1: 0.5919
# Context window: 1
# Loading weights: 100% 199/199 [00:00<00:00, 675.28it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.5772, Train Acc: 0.5403 | Val Loss: 1.4187, Val Acc: 0.5798 Val F1: 0.5529
# Epoch 2 | Train Loss: 1.3097, Train Acc: 0.6227 | Val Loss: 1.3086, Val Acc: 0.6032 Val F1: 0.5865
# Epoch 3 | Train Loss: 1.1940, Train Acc: 0.6458 | Val Loss: 1.2658, Val Acc: 0.5843 Val F1: 0.5864
# Epoch 4 | Train Loss: 1.0810, Train Acc: 0.6788 | Val Loss: 1.2908, Val Acc: 0.5960 Val F1: 0.5983
# Epoch 5 | Train Loss: 0.9577, Train Acc: 0.7068 | Val Loss: 1.3693, Val Acc: 0.6087 Val F1: 0.6081
# Context window: 3
# Loading weights: 100% 199/199 [00:00<00:00, 622.94it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.6002, Train Acc: 0.5307 | Val Loss: 1.3643, Val Acc: 0.5789 Val F1: 0.5591
# Epoch 2 | Train Loss: 1.3069, Train Acc: 0.6243 | Val Loss: 1.3418, Val Acc: 0.6014 Val F1: 0.5816
# Epoch 3 | Train Loss: 1.1765, Train Acc: 0.6587 | Val Loss: 1.2377, Val Acc: 0.6159 Val F1: 0.6184
# Epoch 4 | Train Loss: 1.0611, Train Acc: 0.6879 | Val Loss: 1.2807, Val Acc: 0.5996 Val F1: 0.5974
# Epoch 5 | Train Loss: 0.9365, Train Acc: 0.7152 | Val Loss: 1.3097, Val Acc: 0.6132 Val F1: 0.6154
# Context window: 5
# Loading weights: 100% 199/199 [00:00<00:00, 646.73it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.6144, Train Acc: 0.5282 | Val Loss: 1.3843, Val Acc: 0.5753 Val F1: 0.5554
# Epoch 2 | Train Loss: 1.3036, Train Acc: 0.6266 | Val Loss: 1.3362, Val Acc: 0.6105 Val F1: 0.5932
# Epoch 3 | Train Loss: 1.1809, Train Acc: 0.6541 | Val Loss: 1.2614, Val Acc: 0.6159 Val F1: 0.6100
# Epoch 4 | Train Loss: 1.0556, Train Acc: 0.6847 | Val Loss: 1.2409, Val Acc: 0.6050 Val F1: 0.6037
# Epoch 5 | Train Loss: 0.9474, Train Acc: 0.7173 | Val Loss: 1.2991, Val Acc: 0.5933 Val F1: 0.5990


In [ ]:
#  Context window: 0
# Loading weights: 100% 199/199 [00:00<00:00, 597.40it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.5173, Train Acc: 0.5782 | Val Loss: 1.3559, Val Acc: 0.5924 Val F1: 0.5788
# Epoch 2 | Train Loss: 1.3081, Train Acc: 0.6296 | Val Loss: 1.3696, Val Acc: 0.6050 Val F1: 0.5888
# Epoch 3 | Train Loss: 1.2030, Train Acc: 0.6556 | Val Loss: 1.3030, Val Acc: 0.6041 Val F1: 0.5980
# Epoch 4 | Train Loss: 1.1193, Train Acc: 0.6725 | Val Loss: 1.3237, Val Acc: 0.6032 Val F1: 0.6018
# Epoch 5 | Train Loss: 1.0097, Train Acc: 0.6980 | Val Loss: 1.3531, Val Acc: 0.5933 Val F1: 0.5910

# Best Validation F1 score: 0.6018

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.75      0.78      0.77       470
#     surprise       0.59      0.56      0.57       150
#         fear       0.27      0.30      0.28        40
#      sadness       0.42      0.32      0.37       111
#          joy       0.58      0.57      0.57       163
#      disgust       0.21      0.36      0.27        22
#        anger       0.46      0.44      0.45       153

#     accuracy                           0.60      1109
#    macro avg       0.47      0.48      0.47      1109
# weighted avg       0.60      0.60      0.60      1109

# Context window: 1
# Loading weights: 100% 199/199 [00:00<00:00, 580.48it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.5863, Train Acc: 0.5407 | Val Loss: 1.3735, Val Acc: 0.5735 Val F1: 0.5544
# Epoch 2 | Train Loss: 1.3163, Train Acc: 0.6224 | Val Loss: 1.3300, Val Acc: 0.6087 Val F1: 0.5946
# Epoch 3 | Train Loss: 1.1942, Train Acc: 0.6524 | Val Loss: 1.3080, Val Acc: 0.5906 Val F1: 0.5910
# Epoch 4 | Train Loss: 1.0951, Train Acc: 0.6755 | Val Loss: 1.2850, Val Acc: 0.5807 Val F1: 0.5935
# Epoch 5 | Train Loss: 0.9732, Train Acc: 0.7055 | Val Loss: 1.3107, Val Acc: 0.5951 Val F1: 0.6062

# Best Validation F1 score: 0.6062

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.78      0.72      0.75       470
#     surprise       0.65      0.62      0.63       150
#         fear       0.25      0.35      0.29        40
#      sadness       0.45      0.38      0.41       111
#          joy       0.57      0.56      0.56       163
#      disgust       0.14      0.41      0.21        22
#        anger       0.47      0.48      0.47       153

#     accuracy                           0.60      1109
#    macro avg       0.47      0.50      0.47      1109
# weighted avg       0.62      0.60      0.61      1109

# Context window: 5
# Loading weights: 100% 199/199 [00:00<00:00, 638.88it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.6217, Train Acc: 0.5201 | Val Loss: 1.3777, Val Acc: 0.5708 Val F1: 0.5602
# Epoch 2 | Train Loss: 1.3004, Train Acc: 0.6234 | Val Loss: 1.3128, Val Acc: 0.5978 Val F1: 0.5917
# Epoch 3 | Train Loss: 1.1733, Train Acc: 0.6503 | Val Loss: 1.2502, Val Acc: 0.6014 Val F1: 0.6038
# Epoch 4 | Train Loss: 1.0604, Train Acc: 0.6781 | Val Loss: 1.2800, Val Acc: 0.6267 Val F1: 0.6188
# Epoch 5 | Train Loss: 0.9290, Train Acc: 0.7137 | Val Loss: 1.4130, Val Acc: 0.6150 Val F1: 0.6079

# Best Validation F1 score: 0.6188

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.73      0.83      0.78       470
#     surprise       0.60      0.64      0.62       150
#         fear       0.26      0.35      0.30        40
#      sadness       0.53      0.39      0.45       111
#          joy       0.58      0.51      0.54       163
#      disgust       0.30      0.27      0.29        22
#        anger       0.54      0.42      0.48       153

#     accuracy                           0.63      1109
#    macro avg       0.51      0.49      0.49      1109
# weighted avg       0.62      0.63      0.62      1109

# Context window: 9
# Loading weights: 100% 199/199 [00:00<00:00, 619.35it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 1.6116, Train Acc: 0.5284 | Val Loss: 1.4041, Val Acc: 0.5951 Val F1: 0.5593
# Epoch 2 | Train Loss: 1.3091, Train Acc: 0.6237 | Val Loss: 1.3573, Val Acc: 0.6114 Val F1: 0.5925
# Epoch 3 | Train Loss: 1.1849, Train Acc: 0.6539 | Val Loss: 1.2701, Val Acc: 0.6150 Val F1: 0.6193
# Epoch 4 | Train Loss: 1.0781, Train Acc: 0.6817 | Val Loss: 1.2908, Val Acc: 0.6096 Val F1: 0.6070
# Epoch 5 | Train Loss: 0.9562, Train Acc: 0.7066 | Val Loss: 1.3462, Val Acc: 0.6204 Val F1: 0.6182

# Best Validation F1 score: 0.6193

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.77      0.77      0.77       470
#     surprise       0.62      0.62      0.62       150
#         fear       0.35      0.30      0.32        40
#      sadness       0.41      0.50      0.45       111
#          joy       0.65      0.49      0.56       163
#      disgust       0.19      0.45      0.26        22
#        anger       0.49      0.46      0.48       153

#     accuracy                           0.61      1109
#    macro avg       0.50      0.51      0.49      1109
# weighted avg       0.63      0.61      0.62      1109



In [ ]:
#  Context window: 0
# Loading weights: 100% 199/199 [00:00<00:00, 387.48it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 0.8984, Train Acc: 0.5839 | Val Loss: 0.8053, Val Acc: 0.6078 Val F1: 0.5729
# Epoch 2 | Train Loss: 0.7487, Train Acc: 0.6433 | Val Loss: 0.8106, Val Acc: 0.6294 Val F1: 0.5836
# Epoch 3 | Train Loss: 0.6905, Train Acc: 0.6655 | Val Loss: 0.7863, Val Acc: 0.6258 Val F1: 0.5996
# Epoch 4 | Train Loss: 0.6233, Train Acc: 0.6864 | Val Loss: 0.7846, Val Acc: 0.6123 Val F1: 0.5848
# Epoch 5 | Train Loss: 0.5616, Train Acc: 0.7140 | Val Loss: 0.7989, Val Acc: 0.6177 Val F1: 0.6050
# Epoch 6 | Train Loss: 0.4895, Train Acc: 0.7405 | Val Loss: 0.8459, Val Acc: 0.6204 Val F1: 0.6017
# Epoch 7 | Train Loss: 0.4197, Train Acc: 0.7688 | Val Loss: 0.9159, Val Acc: 0.6213 Val F1: 0.6008

# Best Validation F1 score: 0.6050

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.74      0.81      0.77       470
#     surprise       0.59      0.53      0.56       150
#         fear       0.43      0.23      0.30        40
#      sadness       0.43      0.26      0.33       111
#          joy       0.54      0.63      0.58       163
#      disgust       0.40      0.18      0.25        22
#        anger       0.47      0.52      0.50       153

#     accuracy                           0.62      1109
#    macro avg       0.51      0.45      0.47      1109
# weighted avg       0.60      0.62      0.60      1109

# Context window: 1
# Loading weights: 100% 199/199 [00:00<00:00, 655.89it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 0.9647, Train Acc: 0.5506 | Val Loss: 0.8540, Val Acc: 0.5852 Val F1: 0.5363
# Epoch 2 | Train Loss: 0.7671, Train Acc: 0.6340 | Val Loss: 0.7726, Val Acc: 0.6096 Val F1: 0.5769
# Epoch 3 | Train Loss: 0.6878, Train Acc: 0.6678 | Val Loss: 0.7810, Val Acc: 0.6222 Val F1: 0.5803
# Epoch 4 | Train Loss: 0.6167, Train Acc: 0.6901 | Val Loss: 0.7574, Val Acc: 0.6276 Val F1: 0.5993
# Epoch 5 | Train Loss: 0.5447, Train Acc: 0.7201 | Val Loss: 0.7550, Val Acc: 0.6447 Val F1: 0.6273
# Epoch 6 | Train Loss: 0.4683, Train Acc: 0.7491 | Val Loss: 0.8372, Val Acc: 0.6186 Val F1: 0.6137
# Epoch 7 | Train Loss: 0.3852, Train Acc: 0.7877 | Val Loss: 0.8567, Val Acc: 0.6249 Val F1: 0.6131

# Best Validation F1 score: 0.6273

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.71      0.88      0.78       470
#     surprise       0.61      0.64      0.63       150
#         fear       0.42      0.25      0.31        40
#      sadness       0.54      0.39      0.45       111
#          joy       0.60      0.53      0.56       163
#      disgust       0.38      0.23      0.29        22
#        anger       0.56      0.41      0.47       153

#     accuracy                           0.64      1109
#    macro avg       0.55      0.47      0.50      1109
# weighted avg       0.63      0.64      0.63      1109

# Context window: 3
# Loading weights: 100% 199/199 [00:00<00:00, 627.77it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 0.9610, Train Acc: 0.5453 | Val Loss: 0.8292, Val Acc: 0.5906 Val F1: 0.5535
# Epoch 2 | Train Loss: 0.7554, Train Acc: 0.6373 | Val Loss: 0.7771, Val Acc: 0.6204 Val F1: 0.5844
# Epoch 3 | Train Loss: 0.6732, Train Acc: 0.6733 | Val Loss: 0.7368, Val Acc: 0.6321 Val F1: 0.6018
# Epoch 4 | Train Loss: 0.6012, Train Acc: 0.6939 | Val Loss: 0.7698, Val Acc: 0.6438 Val F1: 0.6132
# Epoch 5 | Train Loss: 0.5198, Train Acc: 0.7315 | Val Loss: 0.7744, Val Acc: 0.6050 Val F1: 0.5972
# Epoch 6 | Train Loss: 0.4439, Train Acc: 0.7568 | Val Loss: 0.8060, Val Acc: 0.6303 Val F1: 0.6151
# Epoch 7 | Train Loss: 0.3796, Train Acc: 0.7867 | Val Loss: 0.9832, Val Acc: 0.6204 Val F1: 0.6006

# Best Validation F1 score: 0.6151

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.71      0.84      0.77       470
#     surprise       0.59      0.65      0.62       150
#         fear       0.46      0.28      0.34        40
#      sadness       0.55      0.36      0.43       111
#          joy       0.58      0.52      0.55       163
#      disgust       0.15      0.09      0.11        22
#        anger       0.54      0.44      0.49       153

#     accuracy                           0.63      1109
#    macro avg       0.51      0.46      0.47      1109
# weighted avg       0.61      0.63      0.62      1109

# Context window: 5
# Loading weights: 100% 199/199 [00:00<00:00, 644.62it/s, Materializing param=pooler.dense.weight]BertModel LOAD REPORT from: bert-base-uncased
# Key                                        | Status     |  |
# -------------------------------------------+------------+--+-
# cls.seq_relationship.bias                  | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |
# cls.predictions.transform.dense.weight     | UNEXPECTED |  |
# cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
# cls.predictions.bias                       | UNEXPECTED |  |
# cls.seq_relationship.weight                | UNEXPECTED |  |
# cls.predictions.transform.dense.bias       | UNEXPECTED |  |

# Notes:
# - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
# Epoch 1 | Train Loss: 0.9790, Train Acc: 0.5329 | Val Loss: 0.8288, Val Acc: 0.5807 Val F1: 0.5367
# Epoch 2 | Train Loss: 0.7627, Train Acc: 0.6358 | Val Loss: 0.7623, Val Acc: 0.6231 Val F1: 0.5900
# Epoch 3 | Train Loss: 0.6743, Train Acc: 0.6688 | Val Loss: 0.7403, Val Acc: 0.6240 Val F1: 0.5988
# Epoch 4 | Train Loss: 0.6004, Train Acc: 0.6970 | Val Loss: 0.7414, Val Acc: 0.6375 Val F1: 0.6156
# Epoch 5 | Train Loss: 0.5215, Train Acc: 0.7297 | Val Loss: 0.7378, Val Acc: 0.6231 Val F1: 0.6090
# Epoch 6 | Train Loss: 0.4469, Train Acc: 0.7531 | Val Loss: 0.8125, Val Acc: 0.6195 Val F1: 0.6056
# Epoch 7 | Train Loss: 0.3774, Train Acc: 0.7865 | Val Loss: 0.8869, Val Acc: 0.6168 Val F1: 0.5998

# Best Validation F1 score: 0.6156

# Classification Report (Validation Set):

#               precision    recall  f1-score   support

#      neutral       0.71      0.86      0.78       470
#     surprise       0.60      0.72      0.65       150
#         fear       0.50      0.25      0.33        40
#      sadness       0.51      0.29      0.37       111
#          joy       0.60      0.52      0.55       163
#      disgust       0.50      0.05      0.08        22
#        anger       0.50      0.45      0.48       153

#     accuracy                           0.64      1109
#    macro avg       0.56      0.45      0.46      1109
# weighted avg       0.62      0.64      0.62      1109

